# Consistency Check: transaction.csv vs data_normalise.csv

This notebook compares value/format consistency between source and normalised data.

Checks included:
- Time/date formats
- Distinct values for city, card type, status
- Amount formats
- Side-by-side comparison tables


In [16]:
import re
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 260)

trx = pd.read_csv('transaction.csv', keep_default_na=False)
norm = pd.read_csv('data_normalise.csv', keep_default_na=False)

print(f'transaction.csv shape: {trx.shape}')
print(f'data_normalise.csv shape: {norm.shape}')


transaction.csv shape: (10103, 6)
data_normalise.csv shape: (10103, 7)


In [17]:
# Helper functions
def is_null_like(v):
    if pd.isna(v):
        return True
    s = str(v).strip()
    return s == '' or s.upper() == 'NULL'

def classify_transaction_time(s):
    t = str(s).strip()
    if is_null_like(t):
        return 'NULL/empty'

    pats = [
        ('HH:MM YYYY-MM-DD', r'^\d{1,2}:\d{2}\s+\d{4}-\d{2}-\d{2}$'),
        ('HH:MM:SS YYYY-MM-DD', r'^\d{1,2}:\d{2}:\d{2}\s+\d{4}-\d{2}-\d{2}$'),
        ('YYYY-MM-DD HH:MM', r'^\d{4}-\d{2}-\d{2}\s+\d{1,2}:\d{2}$'),
        ('YYYY-MM-DD HH:MM:SS', r'^\d{4}-\d{2}-\d{2}\s+\d{1,2}:\d{2}:\d{2}$'),
        ('YYYY/MM/DD HH:MM', r'^\d{4}/\d{2}/\d{2}\s+\d{1,2}:\d{2}$'),
        ('YYYY/MM/DD HH:MM:SS', r'^\d{4}/\d{2}/\d{2}\s+\d{1,2}:\d{2}:\d{2}$'),
        ('YYYY-MM-DD only', r'^\d{4}-\d{2}-\d{2}$'),
        ('YYYY/MM/DD only', r'^\d{4}/\d{2}/\d{2}$'),
        ('HH:MM only', r'^\d{1,2}:\d{2}$'),
        ('HH:MM:SS only', r'^\d{1,2}:\d{2}:\d{2}$'),
        ('NULL YYYY-MM-DD', r'^NULL\s+\d{4}-\d{2}-\d{2}$'),
        ('HH:MM NULL', r'^\d{1,2}:\d{2}\s+NULL$'),
    ]
    for name, p in pats:
        if re.match(p, t, flags=re.IGNORECASE):
            return name
    return 'OTHER'

def classify_norm_date(s):
    t = str(s).strip()
    if is_null_like(t):
        return 'NULL/empty'
    return 'YYYY-MM-DD' if re.match(r'^\d{4}-\d{2}-\d{2}$', t) else 'OTHER'

def classify_norm_time(s):
    t = str(s).strip()
    if is_null_like(t):
        return 'NULL/empty'
    return 'HH:MM:SS' if re.match(r'^\d{2}:\d{2}:\d{2}$', t) else 'OTHER'

def classify_amount_format(s):
    t = str(s).strip()
    if is_null_like(t):
        return 'NULL/empty'
    if re.match(r'^-?\d+$', t):
        return 'integer'
    if re.match(r'^-?\d+\.\d+$', t):
        return 'decimal'
    if re.match(r'^-?\d{1,3}(,\d{3})+(\.\d+)?$', t):
        return 'comma-number'
    return 'text/other'


## 1) Time/Date Format Comparison


In [18]:
trx_time_fmt = trx['time'].map(classify_transaction_time).value_counts(dropna=False).rename_axis('format').reset_index(name='count')
trx_time_fmt['dataset'] = 'transaction.time'

norm_time_fmt = norm['time'].map(classify_norm_time).value_counts(dropna=False).rename_axis('format').reset_index(name='count')
norm_time_fmt['dataset'] = 'normalise.time'

norm_date_fmt = norm['date'].map(classify_norm_date).value_counts(dropna=False).rename_axis('format').reset_index(name='count')
norm_date_fmt['dataset'] = 'normalise.date'

time_date_comparison = pd.concat([trx_time_fmt, norm_time_fmt, norm_date_fmt], ignore_index=True)
time_date_comparison = time_date_comparison[['dataset', 'format', 'count']].sort_values(['dataset', 'count'], ascending=[True, False]).reset_index(drop=True)
display(time_date_comparison)


,dataset,format,count
0,normalise.date,YYYY-MM-DD,10080
1,normalise.date,NULL/empty,23
2,normalise.time,HH:MM:SS,10080
3,normalise.time,NULL/empty,23
4,transaction.time,YYYY-MM-DD HH:MM:SS,10000
5,transaction.time,HH:MM YYYY-MM-DD,90
6,transaction.time,NULL/empty,8
7,transaction.time,OTHER,4
8,transaction.time,HH:MM only,1


## 2) Distinct City Values Comparison


In [19]:
city_lookup = pd.read_csv('city.csv')
city_name_to_id = dict(zip(city_lookup['city'], city_lookup['city_id']))

city_norm_map = {
    'tehran': 'Tehran',
    'thr': 'Tehran',
    'thran': 'Tehran',
    'tehr@n': 'Tehran',
    'tabriz': 'Tabriz',
    'isfahan': 'Isfahan',
    'mashhad': 'Mashhad',
    'shiraz': 'Shiraz',
    'qom': 'Qom',
    'karaj': 'Karaj',
    'ahvaz': 'Ahvaz',
}

trx_city = sorted({str(x).strip() for x in trx['city'] if not is_null_like(x)})

rows = []
for raw_city in trx_city:
    canonical_city = city_norm_map.get(raw_city.lower(), 'NULL')
    city_id = city_name_to_id.get(canonical_city, 'NULL') if canonical_city != 'NULL' else 'NULL'
    rows.append({
        'transaction_city': raw_city,
        'canonical_city': canonical_city,
        'normalise_city_id': city_id,
    })

city_cmp = pd.DataFrame(rows)
display(city_cmp)


,transaction_city,canonical_city,normalise_city_id
0,Ahvaz,Ahvaz,city_1
1,Isfahan,Isfahan,city_2
2,Karaj,Karaj,city_3
3,Mashhad,Mashhad,city_4
4,Qom,Qom,city_5
5,Shiraz,Shiraz,city_6
6,TEHRAN,Tehran,city_8
7,THR,Tehran,city_8
8,Tabriz,Tabriz,city_7
9,Tehran,Tehran,city_8


## 3) Distinct Card Type Values Comparison


In [20]:
trx_card = sorted({str(x).strip() for x in trx['card_type'] if not is_null_like(x)})
norm_card = sorted({str(x).strip() for x in norm['card_id'] if not is_null_like(x)})

card_cmp = pd.DataFrame({'transaction_card_type': pd.Series(trx_card), 'normalise_card_id': pd.Series(norm_card)})
display(card_cmp)


,transaction_card_type,normalise_card_id
0,Amex,card_1
1,Discover,card_2
2,MastCard,card_3
3,Master Card,card_4
4,Master-Card,NaN
5,MasterCard,NaN
6,VISA,NaN
7,Visa,NaN
8,Vsa,NaN
9,nan,NaN


## 4) Distinct Status Values Comparison


In [21]:
trx_status = sorted({str(x).strip() for x in trx['status'] if not is_null_like(x)})
norm_status = sorted({str(x).strip() for x in norm['status'] if not is_null_like(x)})

status_cmp = pd.DataFrame({'transaction_status': pd.Series(trx_status), 'normalise_status': pd.Series(norm_status)})
display(status_cmp)


,transaction_status,normalise_status
0,FAIL,failed
1,Success,success
2,fail,NaN
3,failed,NaN
4,succeed,NaN
5,success,NaN


## 5) Amount Format Comparison


In [22]:
trx_amount_fmt = trx['amount'].map(classify_amount_format).value_counts(dropna=False).rename_axis('format').reset_index(name='count')
trx_amount_fmt['dataset'] = 'transaction.amount'

norm_amount_fmt = norm['amount'].map(classify_amount_format).value_counts(dropna=False).rename_axis('format').reset_index(name='count')
norm_amount_fmt['dataset'] = 'normalise.amount'

amount_fmt_cmp = pd.concat([trx_amount_fmt, norm_amount_fmt], ignore_index=True)
amount_fmt_cmp = amount_fmt_cmp[['dataset', 'format', 'count']].sort_values(['dataset', 'count'], ascending=[True, False]).reset_index(drop=True)
display(amount_fmt_cmp)


,dataset,format,count
0,normalise.amount,decimal,9802
1,normalise.amount,NULL/empty,301
2,transaction.amount,decimal,10003
3,transaction.amount,integer,97
4,transaction.amount,NULL/empty,2
5,transaction.amount,text/other,1


## 6) Final Comparison Summary Table


In [23]:
summary = pd.DataFrame([
    {
        'check': 'rows',
        'transaction': len(trx),
        'normalised': len(norm),
    },
    {
        'check': 'time/date valid-format buckets (count)',
        'transaction': trx_time_fmt['format'].nunique(),
        'normalised': norm_time_fmt['format'].nunique() + norm_date_fmt['format'].nunique(),
    },
    {
        'check': 'unique city values',
        'transaction': len(trx_city),
        'normalised': len(norm_city),
    },
    {
        'check': 'unique card values',
        'transaction': len(trx_card),
        'normalised': len(norm_card),
    },
    {
        'check': 'unique status values',
        'transaction': len(trx_status),
        'normalised': len(norm_status),
    },
    {
        'check': 'amount format buckets',
        'transaction': trx_amount_fmt['format'].nunique(),
        'normalised': norm_amount_fmt['format'].nunique(),
    },
])
display(summary)


,check,transaction,normalised
0,rows,10103,10103
1,time/date valid-format buckets (count),5,4
2,unique city values,14,8
3,unique card values,11,4
4,unique status values,6,2
5,amount format buckets,4,2
